<a href="https://colab.research.google.com/github/bergs692/Project4/blob/main/Rec_Sys_HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resources Disclosure

TODO: Update this section with a list of resources you used. OR delete thie text from this to indicate that you used no resources. Especially with modern LLMs it's _very_ easy to lose track of which resources you used, so we advise you to update this list each time (alongside more direct comment of if you're getting a line of code from somewhere)

Examples:
* (chatgpt) "What's the best way to take the average of a pandas data frame split up by values of a column?" -- response indicated showed me the `groupby` and `mean` functions which I used throughout.
* (github colab in VS code) I used the auto-compete to produce some lines of code -- comments were added where this happened.
* I copied the entire file into claud and it wrote all that is seen here.

(obviously that last one is not in-line with the rules -- but if you do it, the ethical thing to do is disclose it and not take credit for the work)


In [1]:
!pip install git+https://github.com/NicolasHug/Surprise.git@fixnp2
# install surprise -- this takes a minute or so... we might not end up using this framework very much, but it's nice to have a _reall_ tookit


  Cloning https://github.com/NicolasHug/Surprise.git (to revision fixnp2) to /tmp/pip-req-build-ozc2jvib
  Running command git clone --filter=blob:none --quiet https://github.com/NicolasHug/Surprise.git /tmp/pip-req-build-ozc2jvib
  Running command git checkout -b fixnp2 --track origin/fixnp2
  Switched to a new branch 'fixnp2'
  Branch 'fixnp2' set up to track remote branch 'fixnp2' from 'origin'.
  Resolved https://github.com/NicolasHug/Surprise.git to commit 935a0ea7609ee10897c5d9c6de9ed18df5fdcd93
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2556282 sha256=8e2bc551c739aca740f3f1beaceabea5737834a19c26948075e4bea107842108
  Stored in directory: /tmp/pip-ephem-wheel-cache-v7hvoztw/wheels/68/ab/41/537c53abc220a1fb4a11000e7272d0478452b472e9f3a9359d
Successfully built scikit-surprise


In [2]:
# install surprise -- this takes a minute or so... we might not end up using this framework very much, but it's nice to have a _reall_ tookit
import random
import numpy as np
import pandas as pd

my_seed = 2025
random.seed(my_seed)
np.random.seed(my_seed)

In [3]:
# use surprise to download the ml-1m dataset (https://grouplens.org/datasets/movielens/1m/). Please input "Y" when being asked "Do you want to download it".
from surprise import Dataset

data = Dataset.load_builtin('ml-1m')

Dataset ml-1m could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-1m.zip...
Done! Dataset ml-1m has been saved to /root/.surprise_data/ml-1m


In [4]:
# Pre-processing step -- filter down to the N most popular users and M most popular items. This will DEFINTIELY skew our end-results since these are the "easy" items and "easy" users to recommend for.
N_ITEMS = 1164
N_USERS = 1812
# These numbers were chosen to keep the ratio of items to users about the same.


# first filter to the popular items
ratings_df = pd.DataFrame(data.raw_ratings, columns=["userId", "movieId", "rating", "tstamp"])
rating_count_by_item = ratings_df.groupby("movieId")["rating"].count()
rating_count_by_item = rating_count_by_item.sort_values(ascending=False)
ITEMS = rating_count_by_item.index[:N_ITEMS]
ratings_df = ratings_df[ratings_df['movieId'].isin(ITEMS)]

# then filter to the active users
rating_count_by_user = ratings_df.groupby("userId")["rating"].count()
rating_count_by_user = rating_count_by_user.sort_values(ascending=False)
USERS = rating_count_by_user.index[:N_USERS]
ratings_df = ratings_df[ratings_df['userId'].isin(USERS)]

# Finally, convert to a matrix structure.
rating_matrix = ratings_df.pivot(index='userId', columns='movieId', values='rating')

# # just for sorting the rows/columns as they were alphabetical before
rating_matrix = rating_matrix.reindex(sorted(rating_matrix.index, key=int), axis=0)
rating_matrix = rating_matrix.reindex(sorted(rating_matrix.columns, key=int), axis=1)

# # user x movie
display(rating_matrix)

movieId,1,2,3,5,6,7,10,11,16,17,...,3873,3893,3897,3911,3916,3917,3927,3948,3949,3952
userId,,,,,,,,,,,,,,,,,,,,,
10,5.0,5.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN
15,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN
18,4.0,2.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,4.0,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
19,5.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6025,5.0,NaN,3.0,NaN,NaN,4.0,NaN,4.0,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6035,4.0,NaN,1.0,1.0,NaN,3.0,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6036,NaN,NaN,NaN,NaN,3.0,NaN,NaN,3.0,3.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import os

# Locate the movies.dat file using the path from the surprise dataset
ratings_file = data.ratings_file
movies_file = os.path.join(os.path.dirname(ratings_file), "movies.dat")

# Load the movies data
items = pd.read_csv(movies_file,
                    delimiter="::",
                    names=["movieId", "movieName", "genres"],
                    engine='python',
                    encoding="ISO-8859-1")

# Convert movieId to string to match the column names in rating_matrix
items['movieId'] = items['movieId'].astype(str)

# Filter to include only the N most popular items (ITEMS) defined in pre-processing
items = items[items['movieId'].isin(ITEMS)]
items = items.sort_values('movieId').reset_index(drop=True)

# Create the genre matrix
genre_matrix = items['genres'].str.get_dummies(sep='|')
# Set the index to be the movieId so we can align with ratings
genre_matrix.index = items['movieId']

display("items dataframe")
display(items.head())
display("genre_matrix dataframe")
display(genre_matrix.head())
# Note -- it's not clear in advance -- but the movieIds are _strings_ here.


'items dataframe'

,movieId,movieName,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,10,GoldenEye (1995),Action|Adventure|Thriller
2,1009,Escape to Witch Mountain (1975),Adventure|Children's|Fantasy
3,1012,Old Yeller (1957),Children's|Drama
4,1017,Swiss Family Robinson (1960),Adventure|Children's


'genre_matrix dataframe'

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
10,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1009,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
1012,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1017,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Last step -- you'll want to use the datasets "data" -- we need this to have the setup filter based on the earlier data.

data = Dataset.load_from_df(ratings_df[["userId", "movieId", "rating"]], data.reader)

## Some notes on working with Surprise Algorithms

* [General guide](https://surprise.readthedocs.io/en/stable/building_custom_algo.html) This page jumps right into "building algorithms" -- but you can click around the docs from here easily.
* [Important note on IDs](https://surprise.readthedocs.io/en/stable/FAQ.html#raw-inner-note) -- Very short version -- a lot of real-world datasets have real-world dataset problems, such as item- and user-ids that are spread out, or non-numeric (making them insuitable for indexing arrays and matrixes). As such surprise has two SERPATE domains for user- and item-ids "raw" (string-valued dataset) ids, and "internal" (sequential integer) ids. The internal ids are nicely lined up and perfect for indexing into arrays and matrixes.

To help get the ball rolling here are two examples which can serve as basis for the work you will do later. This included one prediction algorithm that always predict the global average rating, and another that always predicts sci-fi at 5 stars.

In [7]:
# An example of how to work with our data and the surprise objects.
from surprise import AlgoBase

# Taken from surprise docs, udpated for a pandas conversion
# (not super-necissary -- the numpy version in the tutorial is probably faster)
# and to show off internal/raw id conversion
class GlobalAveragePredictor(AlgoBase):
    def __init__(self):

        # Always call base method before doing anything.
        AlgoBase.__init__(self)

    def fit(self, trainset):

        # Here again: call base method before doing anything.
        AlgoBase.fit(self, trainset)

        # also prepare a pandas dataframe for familiatiry
        # colums are (internal) user_id , (internal) item_id and rating value.
        self.ratings_df =  pd.DataFrame(self.trainset.all_ratings(), columns=['uid', 'iid', 'rating'])

        # Compute the average rating.
        self.the_mean = self.ratings_df['rating'].mean()

        return self

    def estimate(self, u, i):
        return self.the_mean


# A second example here. This one uses our genre data -- recommending all "sci-fi" at 5, and everything else at a configurable default.
class SciFiFan(AlgoBase):
    def __init__(self, default_pred): # values passed to the `__init__` function are thought of as meta-parameters.
        AlgoBase.__init__(self)
        self.default_pred = default_pred

    # We do not override fit here because we don't do anything special to train the model.

    def estimate(self, u, i):
        # NOTE u and i are internal ids
        if self.trainset.knows_item(i):
            ml_item_id = self.trainset.to_raw_iid(i) # This let's us convert internal item ids to raw item ids.
            if genre_matrix.loc[ml_item_id, 'Sci-Fi']:
              #https://surprise.readthedocs.io/en/stable/trainset.html#surprise.Trainset.rating_scale
              return self.trainset.rating_scale[1]
            else:
              return self.default_pred
        else: # unknown item
          return self.default_pred

In [8]:
# And here's an example using built-in surprise evaluation code to see how these do.
# I should note -- there are better evaluations you can build in surprise than this.
# Example: https://surprise.readthedocs.io/en/stable/getting_started.html#use-cross-validation-iterators

# This example just does a basic 25% hold-out train-test split and uses RMSE to evaluate.

from surprise.model_selection import train_test_split
from surprise import accuracy

# sample random trainset (75% of ratings) and testset (25% of ratings)
trainset, testset = train_test_split(data, test_size=0.25)

# define algorithms
avg_algo = GlobalAveragePredictor()
fan_algo_2 = SciFiFan(2) # if not sci-fi predict at 2
fan_algo_0 = SciFiFan(0) # if not sci-fi predict at 0

results = []
avg_algo.fit(trainset)
results.append(("avg_aglo", accuracy.rmse(avg_algo.test(testset),verbose=False)))

fan_algo_2.fit(trainset)
results.append(("fan_algo_2", accuracy.rmse(fan_algo_2.test(testset),verbose=False)))

fan_algo_0.fit(trainset)
results.append(("fan_algo_0", accuracy.rmse(fan_algo_0.test(testset), verbose=False)))

results_df = pd.DataFrame(results, columns=("algorithm", "RMSE"))
display(results_df)
# As you might expect -- the sci-fi fan algorithm isn't actually very good for most users, so we have pretty big errors on average.


,algorithm,RMSE
0,avg_aglo,1.072593
1,fan_algo_2,1.963540
2,fan_algo_0,2.722569


# Algorithm 1: User-Item Average.

This algorithm is _barely_ a personalized algorithm -- while it does make different predictions for each user, it will ultimatley provide the same recommendation to every user. The prediction equation for this algorithm is as follows:

$\hat{r_{ui}} = \mu_i + \mu_u$

In otherwords -- to predict for user $u$ and item $i$ we add an item-score and a user-score. These two scores can be computed MANY ways, but the most common is a two-pass algorithm:

1. Given the list of training ratings...
2. compute the global average rating across all items and users
3. compute a damped item-average for each item. (damp the averages to the global mean) -- these values are $\mu_i$
4. substract the average item ratings from the observed ratings
5. With these modified ratings compute the user-average rating for each user (again damped, in this case damp the ratings towards 0) -- these are the $\mu_u$ values, They are an estimate of how much a user tends to rate above, or below, the average for the items they have rated.

Then when predicting we can:
1. look up the $mu_u$ and $mu_i$ values and compute $\hat{r_{ui}}$
2. If the user was not in our training dataset, we can use $\mu_u = 0$.
3. If the item was not in our training dataset, we can use $\mu_i = \mu_g$ (where $\mu_g$ is the global average rating)


In [9]:
class UserItemBaseline(AlgoBase):
    def __init__(self, damping_factor=5):
        AlgoBase.__init__(self)
        self.damping_factor = damping_factor




    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        # 1. delete or reset storage for mu_g, mu_u and mu_i values
        # (algorithm objects are meant to be trained repeatedly, and should not carry data over)
        self.mu_g = None
        self.mu_u = None
        self.mu_i = None

        # 2. compute mu_g -- the global average
        self.mu_g = np.mean([r for ir in self.trainset.ir.values() for (_, r) in ir])
        self.n_tI = len(self.trainset.ir)

        # 3. compute mu_i -- the damped item-averages
        self.mu_i = {
            i: (
             ((len(item) * np.mean([r for (_,r) in item]) + (self.damping_factor * self.mu_g)) / (len(item) + self.damping_factor))
            )
            for i, item in self.trainset.ir.items()
        }

        # 4. compute mu_u -- the damped user-averages
        self.mu_u = {
            u:
            (
             ((len(user) * np.mean([r for (_,r) in user]) + (self.damping_factor * self.mu_g)) / (len(user) + self.damping_factor))
            )
            for u, user in self.trainset.ur.items()
        }

        # This function returns self.
        return self

    def estimate_all_mu_u(self, i):
      return [(u, self.estimate(u, i)) for u in self.trainset.all_users()]


    def estimateAll_mu_i(self, u):
      return True


    def estimate(self, u, i, silence=True):
        # pull up mu_u and mu_i -- or if the items are not known to the training dataset -- fallback values
        u_fall = u if self.trainset.knows_user(u) else -1
        i_fall = i if self.trainset.knows_item(i) else -1


        if u_fall == 0 and not silence: print('User Fallback')
        if i_fall == -1 and not silence: print('Item Fallback')


        mu_i = self.mu_i[i_fall] if i_fall != -1 else self.mu_g

        mu_u = self.mu_u[u_fall] if u_fall != -1 else 0


        return mu_u + mu_i


## Part 1 Questions


In [10]:
trainset1, testset1 = train_test_split(data, test_size=0.25)

In [11]:
# What would we predict for user 175 and item 1 using this baseline model.

UIB = UserItemBaseline()
UIB.fit(trainset1)

print("What would we predict for user 175 and item 1 using this baseline model: ", UIB.estimate(175,1))


What would we predict for user 175 and item 1 using this baseline model:  7.389915828272244


In [12]:
fullData = data.build_full_trainset()

In [13]:
# Who are the users with the top-5 and bottom-5 values for mu_u? What are those values?

#for user_id in trainset1.all_users():


bottom_results = [user for (user,_) in sorted(UIB.estimate_all_mu_u(1), key = lambda t: t[1])[0:5]]

print('(Training Data) Bottom 5 Users (bottom first)', bottom_results)

top_results = [user for (user,_) in sorted(UIB.estimate_all_mu_u(1), key = lambda t: t[1],reverse=True)[0:5]]

print('(Training Data) Top 5 Users (top first)', top_results)


print()
print()

#(Hint -- the following code will train the model on all of the data -- you can then inspect the fit instance data.)
temp  = UserItemBaseline()
temp.fit(fullData)

bottom_results = [user for (user,_) in sorted(temp.estimate_all_mu_u(1), key = lambda t: t[1])[0:5]]

print('(Full Data) Bottom 5 Users (bottom first)', bottom_results)

top_results = [user for (user,_) in sorted(temp.estimate_all_mu_u(1), key = lambda t: t[1],reverse=True)[0:5]]

print('(Full Data) Top 5 Users (top first)', top_results)

(Training Data) Bottom 5 Users (bottom first) [477, 479, 657, 1453, 682]
(Training Data) Top 5 Users (top first) [1799, 1707, 1736, 1134, 1694]


(Full Data) Bottom 5 Users (bottom first) [1705, 1508, 931, 415, 387]
(Full Data) Top 5 Users (top first) [1428, 255, 1220, 1763, 1462]


In [14]:
# What does a high or low value of mu_u mean about a user?
# Lower mu_u value for a user means they rate items (movies) lower on average compared to someone with a higher mu_u value.
# So then user 569 (inner id) on a damped average, rates items higher than all other users.
print('(Full Data) Top 5 Users (top first) with Values: ',[sorted(temp.estimate_all_mu_u(1), key = lambda t: t[1],reverse=True)[0:5]])

(Full Data) Top 5 Users (top first) with Values:  [[(1428, np.float64(8.105306564317964)), (255, np.float64(8.07862318136961)), (1220, np.float64(7.999075188432281)), (1763, np.float64(7.980173969737841)), (1462, np.float64(7.95594983683803))]]


# Algorithm 2: Item-Item Collaborative Filtering
For this part you should implement the Item-Item collaborative filtering algorithm. Because this algorithm often is deployed with a pre-computed and truncated similarity model, we'll do the same here.

The constructor takes parameters:

* `model_size` -- When training the algorithm, how many possible neighbors to keep for each item? (keep the top `model_size` most similar items when training the model)
* `neighborhood_size` -- How many neighbors to use when actually predicting.


## Train (fit function)
Given the ratings this should:

1. Compute the global average rating.
2. Compute user-average ratings for the next step (use a damped average with damping factor of 5.
3. compute the item-item similarity matrix using adjusted cosine similarity. $$S_{ik} = \frac{ \sum_{u \in U_i \cap U_k} (r_{ui}-\mu_u)(r_{uk}-\mu_u)}{\sqrt{\sum_{u \in U_i \cap U_k} (r_{ui}-\mu_u)^2} \sqrt{\sum_{u \in U_i \cap U_k} (r_{uk}-\mu_u)^2}}$$  
4. create the "truncated model" for item-item. As a reminder we store this instead of a complete item-item similarty matrix. You can choose how best to store this, but it should contain only `model_size` most similar items for each item (as well as their similarity scores).
    * Side note -- do not include an item as it's own potential neghbor.
5. You may store any other data you may want in advance. For example, you may want to pre-compute sets or lists of which items each user has rated.

## Predict (estimate)
This takes an internal user_id and an internal item_id and:

1. Figures out which items in the truncated model the user has rated.
2. Compute the neighborhood of the items-to-predict (`newighborhood_size` most similar rated items)
3. Computes the prediction using the equation from class: $$\hat{r}_{ui} = \frac{\sum_{k \in I_u \cap N_i}S_{ik}r_{uk}}{\sum_{k \in I_u \cap N_i}|S_{ik}|}$$

## Some hints:

Efficient computation of the item-item similarit matrix is important -- but tricky. We've downsized the dataset so a less efficient loop-based solution will probably be _fast enough to not be terrible_. Try to begin by coding something you're positive (and can confirm) is correct -- then loop back and make it faster while keeping the "obviously correct" code around so you can confirm your results are the same.

Please do not use the surprise `compute_similarity` function. That kind of defeats the purpose of this exercise -- you can, however, see if there's any pre-made functions in pandas, sci-kit, or numpy which can solve this problem (possibly with some data pre- or post-processing on your part).

While we have a few "built-in" checks in the questions that follow -- you should be pro-actively checking your computed values here -- it'll be easy to miss bugs if you're not looking at these and checking them against your instincts.

If asked to predict an item or user not known to the dataset use the global average rating as a "fallback" prediction.

In [15]:
import math

In [16]:
class SimCache:
    def __init__(self):
        self.is_valid = False
        self.sim_combos = {}
        self.trainset_hash = None

    def store(self, sim_combos, trainset):
        self.sim_combos = sim_combos
        self.trainset_hash = self._hash_trainset(trainset)
        self.is_valid = True

    def get(self, trainset):
        if self.is_valid and self._hash_trainset(trainset) == self.trainset_hash:
            return self.sim_combos
        return None

    def invalidate(self):
        self.is_valid = False
        self.sim_combos = {}

    def _hash_trainset(self, trainset):
        # Hash based on number of users, items, ratings as a lightweight check
        return (trainset.n_users, trainset.n_items, trainset.n_ratings)

# Create a single global cache instance
sim_cache = SimCache()

In [17]:
from numpy.random.mtrand import f
class ItemItemCF(AlgoBase):
    def __init__(self, model_size = 100, neighborhood_size=5):
        AlgoBase.__init__(self)
        self.model_size = model_size
        self.neighborhood_size = neighborhood_size

    def get_users_who_rated_both(self, item1_id, item2_id):
      ratings_1 = self.trainset.ir[item1_id]
      ratings_2 = self.trainset.ir[item2_id]
      return set(id1 for id1, _ in ratings_1) & set(id2 for id2, _ in ratings_2)

    def get_items_with_user_and_neighborhood(self, user_id, movie_id, neighborhood):
      items_in_both = []
      for item in neighborhood:
          item_id = item[0]
          if item_id != movie_id and item_id in self.rating_dict_dict[user_id]:
              items_in_both.append(item)
      return items_in_both

    def compute_all_users_damped_average(self):
      self.user_mu_dict = {
            u:
            (
             ((len(user) * np.mean([r for (_,r) in user]) + (self.damping_strength * self.mu_g)) / (len(user) + self.damping_strength))
            )
            for u, user in self.trainset.ur.items()
        }

    def check_pair_in_dict(self, item1, item2):
      check_key = (min(item1, item2), max(item1, item2))
      return check_key in self.sim_combos

    # (user -> item -> rating)
    def compute_rating_dict_dict(self):
      users = self.trainset.all_users()
      self.rating_dict_dict = {}
      users_ratings_dict = self.trainset.ur
      for user in users:
        user_rating_dict = {}
        for rating in users_ratings_dict[user]:
          item = rating[0]
          rating = rating[1]

          user_rating_dict[item] = rating
        self.rating_dict_dict[user] = user_rating_dict


    def compute_all_combos_simularity_dict(self):
      n = len(self.trainset.all_items())
      r, c = np.triu_indices(n, k=1)
      combos_unique = np.column_stack((np.fromiter(self.trainset.all_items(), dtype=np.int64)[r], np.fromiter(self.trainset.all_items(), dtype=np.int64)[c]))
      combo_dict = {}

      for combo in combos_unique:
        item1 = combo[0]
        item2 = combo[1]
        if item1 != item2 and not self.check_pair_in_dict(item1, item2):
          users = self.get_users_who_rated_both(item1, item2)
          numerator_sum = 0
          denom_sum1 = 0
          denom_sum2 = 0
          if len(users) != 0:
            for user in users:
              temp_user_ratings = self.rating_dict_dict[user]
              i1_r = temp_user_ratings[item1]
              i2_r = temp_user_ratings[item2]
              mu_u = self.user_mu_dict[user]

              numerator_sum += (i1_r - mu_u) * (i2_r - mu_u)
              denom_sum1 += (i1_r - mu_u) ** 2
              denom_sum2 += (i2_r - mu_u) ** 2

            value = (numerator_sum / (np.sqrt(denom_sum1) * np.sqrt(denom_sum2)))
            value_safe = value if not math.isnan(value) else 0 # this would be where all voters align with there damped average exactly and so you would divide by 0
            # Always storing the item1, item2 by id of minimum being first of the ids, makes faster search / check.
            store_key = (min(item1, item2), max(item1, item2))

            combo_dict[store_key] = value_safe
      return combo_dict

    def get_combos_with(self, item_id):
      combos_id = []
      for key, value in self.sim_combos.items():
        i1 = key[0]
        i2 = key[1]
        if i1 == item_id or i2 == item_id:
          id = i1 if i1 != item_id else i2
          combos_id.append([id, value]) # Stripped the parameter id for ret list
      return combos_id

    def get_sim(self, item1, item2):
      minItem = min(item1, item2)
      maxItem = max(item1, item2)
      return self.sim_combos[(minItem, maxItem)]

    def simularity_matrix(self):

      for item_i in list_of_items:
        for item_k in list_of_items:
          if (item_i == item_k):
            continue
          else:
            continue
        np.array(np.meshgrid(a,b)).T.reshape(-1,2)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        self.damping_strength = 5
        # 1. delete or reset all derived values
        self.mu_g = None

        self.user_mu_dict = {}

        self.sim_combos = {}

        self.rating_dict_dict = None

        self.compute_rating_dict_dict()

        # 2. set up any data transformatinos you want
        self.mu_g = np.mean([r for ir in self.trainset.ir.values() for (_, r) in ir])

        self.compute_all_users_damped_average()

        # 3.a Compute the combo matrix simularity
        cached = sim_cache.get(trainset)
        if cached is not None:
            print("Loading sim_combos from cache...")
            self.sim_combos = cached
        else:
            self.sim_combos = self.compute_all_combos_simularity_dict()
            sim_cache.store(self.sim_combos, trainset)

        # 3. Compute the similarity matrix
        #print(self.sim_combos[1:10])
        #for key, value in self.sim_combos.items():
          #if not math.isnan(value):
          #print(f"{key}: {value}")

        # 4. truncate the similarity model and store

        # 5. Any further pre-computation any you want to do should happen here.

        # This function returns self.
        return self
    def movie_tostring(self, inner_id):
      raw_id = self.trainset.to_raw_iid(inner_id)
      result = items[items['movieId'] == raw_id]['movieName']
      return result.values[0] if not result.empty else None

    def neighborhood_tostring(self, movieId, neighborhoodList):
      for neigh in neighborhoodList:
          movieName = self.movie_tostring(neigh[0])
          sim_val = neigh[1]
          print(movieName, " sim score for ",self.movie_tostring(movieId),sim_val)

    def predict_equation(self, u, i, i_neighborhood):
      rated_neighbors = self.get_items_with_user_and_neighborhood(u,i,i_neighborhood)
      num = 0
      denom = 0
      for rated_item in rated_neighbors:
        itemSim = rated_item[1]
        ratingUser = self.rating_dict_dict[u][rated_item[0]]
        num += itemSim*ratingUser
        denom += np.abs(itemSim)
      return num / denom

    def get_movie_neighborhood(self, i):
      inner_i = self.trainset.to_inner_iid(str(i))
      i_sims = self.get_combos_with(inner_i)
      sorted_i_sims = sorted(i_sims, key=lambda x: x[1], reverse=True)
      neighborhood = sorted_i_sims[0:self.neighborhood_size]
      return neighborhood

    def get_movie_similarities_sorted_n(self, i, amount):
      inner_i = self.trainset.to_inner_iid(str(i))
      i_sims = self.get_combos_with(inner_i)
      sorted_i_sims = sorted(i_sims, key=lambda x: x[1], reverse=True)
      if len(sorted_i_sims) < amount:
        amount = len(sorted_i_sims)
      neighborhood = sorted_i_sims[0:amount]
      return neighborhood


    def estimate(self, u, i):
        # 1. compute the neighborhood for this prediction
        # 2. predict
        # Figures out which items in the truncated model the user has rated.
        # Compute the neighborhood of the items-to-predict (newighborhood_size most similar rated items)
        inner_i = self.trainset.to_inner_iid(str(i))
        i_sims = self.get_combos_with(inner_i)
        sorted_i_sims = sorted(i_sims, key=lambda x: x[1], reverse=True)
        neighborhood = sorted_i_sims[0:self.neighborhood_size]

        print("user:", u, "Item: ", i, self.predict_equation(u,inner_i,neighborhood))




        self.neighborhood_tostring(inner_i, neighborhood)
        # Computes the prediction using the equation from class:

        return 1



In [18]:
trainset2, testset2 = train_test_split(data, test_size=0.50)

In [19]:
II = ItemItemCF()
II.fit(trainset2) # Switched to full data so it can be cached with the part 2 questions part below. Each new session will probably take 359 seconds to train (when cache is reset) I think complexity for x (percent to of dataset in training data) = 6.16+182x+171x^{2} (from google sheets estimate of scatter plot trials)


In [20]:
print(II.estimate(16,11))

user: 16 Item:  11 4.0
Raise the Red Lantern (1991)  sim score for  American President, The (1995) 0.6590488187681193
Dave (1993)  sim score for  American President, The (1995) 0.6199063969935784
Roman Holiday (1953)  sim score for  American President, The (1995) 0.5318617504922791
King and I, The (1956)  sim score for  American President, The (1995) 0.5002199186949736
Hamlet (1996)  sim score for  American President, The (1995) 0.4996410171452274
1


## Part 2 Questions
Again -- we're using these questions both as a way to encourage you to check intermediary results for seeming _reasonable_ and a quick way for us to check if your results match those produced by any common solutions.

In [21]:
# Train an item-item model on all ratings in the dataset as seen above, use this data for the following questions
temp2  = ItemItemCF()
temp2.fit(trainset2)

Loading sim_combos from cache...


In [22]:
# What are the 20 most similar items to MOVIE WAS SWITCHED (movieId 11) and what are their similarity scores?
inner_1 = temp2.trainset.to_inner_iid("2959")
print(temp2.neighborhood_tostring(inner_1, temp2.get_movie_similarities_sorted_n(2959, 20)))

Ghost in the Shell (Kokaku kidotai) (1995)  sim score for  Fight Club (1999) 0.6205700652030939
Requiem for a Dream (2000)  sim score for  Fight Club (1999) 0.5820184306747139
Watership Down (1978)  sim score for  Fight Club (1999) 0.5737070922144101
Run Lola Run (Lola rennt) (1998)  sim score for  Fight Club (1999) 0.5678025276010117
Hard-Boiled (Lashou shentan) (1992)  sim score for  Fight Club (1999) 0.5495877856290184
Treasure of the Sierra Madre, The (1948)  sim score for  Fight Club (1999) 0.5448270924173063
South Park: Bigger, Longer and Uncut (1999)  sim score for  Fight Club (1999) 0.5442470713446209
Inherit the Wind (1960)  sim score for  Fight Club (1999) 0.5302212930360612
American History X (1998)  sim score for  Fight Club (1999) 0.5251617861483684
Life Is Beautiful (La Vita è bella) (1997)  sim score for  Fight Club (1999) 0.5233423307645044
It's a Wonderful Life (1946)  sim score for  Fight Club (1999) 0.5220238841176055
Man Who Would Be King, The (1975)  sim score for 

In [23]:
items

,movieId,movieName,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,10,GoldenEye (1995),Action|Adventure|Thriller
2,1009,Escape to Witch Mountain (1975),Adventure|Children's|Fantasy
3,1012,Old Yeller (1957),Children's|Drama
4,1017,Swiss Family Robinson (1960),Adventure|Children's
...,...,...,...
1159,968,Night of the Living Dead (1968),Horror|Sci-Fi
1160,969,"African Queen, The (1951)",Action|Adventure|Romance|War
1161,971,Cat on a Hot Tin Roof (1958),Drama
1162,994,Big Night (1996),Drama


In [24]:
# What does Item-Item predict for user 195 and movieId 1 (toy story)
inner_1 = temp2.trainset.to_inner_iid("1")
temp2.estimate(195, 1)

user: 195 Item:  1 5.0
Toy Story 2 (1999)  sim score for  Toy Story (1995) 0.706006106749024
Grand Day Out, A (1992)  sim score for  Toy Story (1995) 0.6961384980531153
It Happened One Night (1934)  sim score for  Toy Story (1995) 0.6571503882211854
Wrong Trousers, The (1993)  sim score for  Toy Story (1995) 0.5866642791390951
Henry V (1989)  sim score for  Toy Story (1995) 0.5621367449124888


1

In [25]:
# What are user 195's top-10 highest predicted items (excluding those they have rated already)

In [26]:
# Which item (by name please) appears the most in the truncated model?

# Algorithm 3 Content Filtering
For this part we'll implement a basic content-based personalized recommenders. These will be based off the movie genres.

We will implement a basic approach simple-count based approach. This is not particularly elegant, but it gives us something meaningful to compare item-item against.


## Computing Item Models.

We will be modeling the item data directly with the genre matrix. This is pre-computed in the provided data loading part of this notebook. To summarize though, each movie will be represented with an item-model: $$\theta_i = \left< g_{i,\textrm{Action}}, g_{i,\textrm{Adventure}}, \ldots, g_{i,\textrm{War}}, g_{i,\textrm{Western}}\right> $$Where $\theta_i$ is the item model and each $g_{i,\textrm{genre}}$ is 1 if item $i$ has the given genre, and 0 if it does not.

In the provided code we compute this as a pandas dataframe -- you can see an example of how to quickly access it in the example algorithms -- note that it is indexed by the movie id (which would be the external id in surprise, not the raw id)

## Computing user models.

This is done in the fit function. Make sure you use the passed in rating data -- not the rating matrix from earlier in this notebook. (You may find it useful to re-use that code to make a rating matrix for the passed in rating data) If you use "all" rating data, instead of the passed in rating data, your algorithm would technically be "cheating" in it's later evaluation.

For this algorithm, the user model is: $$\theta_u = \left< g_{u,\textrm{Action}}, g_{u,\textrm{Adventure}}, \ldots, g_{u,\textrm{War}}, g_{u,\textrm{Western}}\right> $$Where $\theta_u$ is the user model and each $g_{u,\textrm{genre}}$ is a count of how many movies in that genre user $u$ liked, minus how many they disliked. (Where we say a user "liked" a movie if they rated it 4 or greater, otherwise they disliked it).

You are free to compute this how you will, but here's a rough outline of how Kluver might approach this problem (to get you started):

1. Convert the provided training set into a rating dataframe
2. convert the rating values so that it has the value +1 where $r_{ui} \geq 4$ and -1 where $r_{ui} < 4$
3. Join this data frame against the genres matrix so that you know the genres for each rating.
4. Finally you need to summarize this into the genres. One way to do this would be per-genre:
    * First filter ratings to only those with a given genre (say, "Action")
    * Then do a per-user sum over this -- that will be the $g_{u,\textrm{Action}}$ values for each user.
    * Repeat this for each genre and merge the results together to form a $theta_u$ matrix.

## estimate
Finally, to predict you can simply take the dot product of $\tehta_u$ and $\theta_i$. This will give you a numeric value that, in principal, should be positive for movies the user will like, and negative for those they will not like.

A few notes here:

1. Your item genre matrix is likely indexed by external "raw" ids, not internal ids -- make sure you account for this.
2. The `TrainSet` class's keeps track of which items and users it's seen -- if it isn't aware of an item it will refuse to map to the raw id -- as such you will likely need a fallback for "unknown" users and "unknown" items -- in these cases please return the 0.


In [27]:
class SimpleGenre(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)


    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        # This function returns self.
        return self

    def estimate(self, u, i):

        return 0



## Algorithm 3 Questions

In [28]:
# As before -- please create a version of your algorithm that's trained on all items and answer the following questions

In [29]:
# What is the item-model for Jurassic Park (Movie 480)

In [30]:
# What is the user-model for user 195? (Note -- don't be afraid to go pull the actual user-rating out and check your math here)

In [31]:
# How often does this algorithm (the version fit on all data) accurately estimate if a user liked/disliked a movie?
# (hint -- the AlgoBase class has quite a few ways to get predictions out.
#          If you want to use the esitmate function directly you'll have to
#          capture the trainingset so you can convert to the internal ids.
#          Otherwise -- you can use the predict function (https://surprise.readthedocs.io/en/stable/algobase.html#surprise.prediction_algorithms.algo_base.AlgoBase.predict)
#          or the test function -- but you'll have to do more work processing
#          the output objects as these are not raw numbers.


# Part 4 -- A basic evaluation

For this part we are going to perform a basic evaluation. Our protocol will be very simple, but more than enough for our needs right now.

1. create a random 20% hold out train-test split (that is to say, put 20% of the ratings in the test set, keep 80% for training.
2. Train all of the algorithms in this notebook with the one training set.

    * global average (provided)
    * sci-fi fan (provided)
    * user-item baseline
    * item-item
    * genre filtering.

   For all of the above, use their default parameters if they have parameters (for example, use the default model size in Item-Item)
3. Compute the RMSE metric for each algorithm using the test set ratings.
4. Provide a sumary table/data-frame with one row per item.

You can see code above, or https://surprise.readthedocs.io/en/stable/getting_started.html#train-test-split-example for examples of how to do a simple evaluation like this in surprise.

In [32]:
# TODO -- put code for eval here.

In [33]:
# According to this evaluation, what algorithm performs the best and why?

In [34]:
# Would trust this evaluation if you were building a real system, and why?

In [35]:
# You Genre filter algorithm (presumably) performed quite poorly.
# Why did this happen?
# Do you think this is an accurate estimate of how well the genre filter would act as a recommender (picking items)

# Part 5 -- our first experiment.

It may surprise you to find out that we _actually do_ use the user-item baseline in MovieLens. We rarely use it directly, but instead we often subtract it from all ratings (which can act like a pre-normalization step making later algorithms easier to code, and generally perform better)

Because we use this as a key part of many other algorithms we have a specific interest in what the idea value of the damping factor should be. (This is also nice since it's probably the fastest algorithm so you won't spend forever on this task)

For this Part -- please devise an evaluation protocol (It's OK if you just use the same one from part 4 -- but I want you to have freedom to use other protocols if you want) and then test versions of the user-item baseline algorithmm with a range of damping values --What is the best damping value?

##Question -- What is your Evaluation Protocol?

(double click to edit)

In [36]:
# Run the eval. Report RMSE value for the ranve of damping factors tried, not just the best damping factor

In [37]:
# What was the best damping factor?